# Generate Persian Non-Conditional Sentences (Negative Examples)

This notebook mirrors `generate_conditionals.ipynb` but generates **negative** examples: Persian
sentences about the same topics that do **not** express any conditional relation (no protasis/
apodosis, marked or unmarked). It uses the same OpenAI-compatible chat model ("GPT 6 Luna"), the
same topic/register sampling, and the same JSONL encoding format (`id`, `text`, `tokens`, `tags`,
`spans`, `prompt_variant`), so the output can be concatenated with the positive dataset for
training a conditional-detection classifier or tagger.

Since there is no conditional span to annotate, every token's BIO tag is `O` and `spans` is always
an empty list; `prompt_variant` is fixed to `"negative"` (kept for schema parity with the positive
dataset).

**Requirements**
- `pip install openai` (and optionally `python-dotenv`)
- An API key in the `OPENAI_API_KEY` environment variable (set it in your shell, or in a `.env`
  file next to this notebook — the notebook will try to load it with `python-dotenv` if available).

In [1]:
import json
import os
import random
import re

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

from openai import OpenAI
from tqdm.auto import tqdm

c:\Users\lischkaf\micromamba\envs\nlp-fall-school\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

Same knobs as `generate_conditionals.ipynb`, pointed at a separate output file so the negative
examples don't overwrite the positive ones.

In [2]:
CONFIG = {
    "model_name": os.environ.get("CONDITIONALS_MODEL_NAME", "gpt-6-luna"),
    "base_url": os.environ.get("OPENAI_BASE_URL"),  # None -> default OpenAI endpoint
    "num_examples": 1000,
    "temperature": 1.0,
    "max_retries": 3,
    "random_seed": 42,
    "output_path": "persian_negatives_bio.jsonl",
}

random.seed(CONFIG["random_seed"])

if "OPENAI_API_KEY" not in os.environ:
    raise RuntimeError(
        "Set the OPENAI_API_KEY environment variable before running this notebook "
        "(e.g. in your shell, or in a .env file next to this notebook)."
    )

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"], base_url=CONFIG["base_url"])

## Tokenizer

Identical regex tokenizer to `generate_conditionals.ipynb`, so token boundaries are consistent
across the positive and negative datasets.

In [3]:
TOKEN_PATTERN = re.compile(r"[\w\u200c]+|[^\s\w]", re.UNICODE)


def tokenize_with_offsets(text):
    """Return a list of (token, start, end) for `text`."""
    return [(m.group(), m.start(), m.end()) for m in TOKEN_PATTERN.finditer(text)]

## Prompt and structured output schema

The schema only asks for `text` (no spans, since there is nothing to annotate). The system prompt
explicitly forbids conditional markers (`اگر`, `هرگاه`, `در صورتی که`, `چنانچه`, `به شرطی که`, ...)
and any implicit conditional relation (imperative+result juxtaposition, subjunctive/future mood
implying a hypothetical, etc.) — the same constructions the `unmarked` prompt in
`generate_conditionals.ipynb` is asked to *produce*. Topics and registers are sampled from the
same `TOPICS` / `REGISTERS` lists as the positive-example notebook.

In [4]:
RESPONSE_FORMAT = {
    "type": "json_schema",
    "json_schema": {
        "name": "persian_non_conditional_example",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "text": {
                    "type": "string",
                    "description": "A single Persian sentence (or short passage) that does not contain any conditional construction.",
                },
            },
            "required": ["text"],
            "additionalProperties": False,
        },
    },
}

SYSTEM_PROMPT = (
    "You generate natural Persian (Farsi) training data for a conditional-clause detection task. "
    "Each response must be a single sentence or short passage that does NOT express any "
    "condition -> consequence relation. Do not use any conditional marker word, such as اگر, "
    "هرگاه, در صورتی که, چنانچه, به شرطی که, وگرنه, or similar. Also avoid implicit conditional "
    "constructions such as imperative + result juxtaposition (\"بخواب، حالت بهتر می‌شه\") or "
    "subjunctive/future verb mood implying a hypothetical outcome. Instead, write plain "
    "statements, descriptions, narrations, questions, or commands that stand on their own without "
    "implying any condition. Vary sentence type (declarative, interrogative, imperative), register "
    "(formal/colloquial), and tense/aspect across responses."
)

TOPICS = [
    "اقتصاد و قیمت‌ها", "آب‌وهوا", "سفر", "تحصیل و دانشگاه", "خانواده و روابط",
    "تکنولوژی", "سلامت", "ورزش", "محیط زیست", "کار و شغل", "غذا و آشپزی",
    "ترافیک شهری", "دوستی", "سیاست", "موسیقی و هنر",
]
REGISTERS = ["محاوره‌ای و غیررسمی", "رسمی و نوشتاری"]

In [5]:
def build_messages():
    topic = random.choice(TOPICS)
    register = random.choice(REGISTERS)
    user_prompt = (
        f"یک جمله فارسی بدون هیچ رابطه‌ی شرطی درباره‌ی «{topic}» تولید کن. "
        f"لحن جمله باید {register} باشد."
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    return messages, topic, register

## Generation with validation and retries

`generate_example` calls the model, checks that the returned text doesn't contain an obvious
conditional marker word (as a cheap sanity filter on top of the prompt instructions), and returns
an example in the same shape as `generate_conditionals.ipynb` — `tags` are all `O` and `spans` is
empty since there is no conditional to annotate.

In [6]:
CONDITIONAL_MARKERS = [
    "اگر", "اگه", "هرگاه", "هر گاه", "در صورتی که", "در صورتیکه", "چنانچه",
    "به شرطی که", "به شرطیکه", "وگرنه", "والا",
]


def generate_example(client, config):
    """Call the model, validate the returned text contains no obvious conditional marker, and
    return a fully-annotated negative example. Retries from scratch up to
    config['max_retries'] times on validation failure."""
    last_err = None
    for attempt in range(1, config["max_retries"] + 1):
        try:
            messages, topic, register = build_messages()
            response = client.chat.completions.create(
                model=config["model_name"],
                messages=messages,
                temperature=config["temperature"],
                response_format=RESPONSE_FORMAT,
            )
            data = json.loads(response.choices[0].message.content)
            text = data["text"].strip()
            if not text:
                raise ValueError("empty text returned")

            for marker in CONDITIONAL_MARKERS:
                if marker in text:
                    raise ValueError(f"conditional marker {marker!r} found in generated text")

            tokens_with_offsets = tokenize_with_offsets(text)
            tags = ["O"] * len(tokens_with_offsets)

            return {
                "text": text,
                "tokens": [tok for tok, _s, _e in tokens_with_offsets],
                "tags": tags,
                "spans": [],
                "prompt_variant": "negative",
                "system_prompt": SYSTEM_PROMPT,
                "topic": topic,
                "register": register,
            }
        except Exception as exc:  # noqa: BLE001 - broad on purpose, we retry
            last_err = exc
            print(f"  [retry {attempt}/{config['max_retries']}] {type(exc).__name__}: {exc}")

    raise RuntimeError(
        f"failed to generate a valid negative example after {config['max_retries']} attempts"
    ) from last_err

In [7]:
examples = []
for i in tqdm(range(CONFIG["num_examples"])):
    try:
        example = generate_example(client, CONFIG)
        example["id"] = i
        examples.append(example)
    except RuntimeError as exc:
        print(f"Skipping example {i}: {exc}")

print(f"Generated {len(examples)}/{CONFIG['num_examples']} examples")

100%|██████████| 1000/1000 [25:45<00:00,  1.55s/it]

Generated 1000/1000 examples


## Inspect a sample

In [8]:
def show_example(example):
    print(f"[{example['prompt_variant']}] ({example['topic']}, {example['register']}) {example['text']}")
    for tok, tag in zip(example["tokens"], example["tags"]):
        print(f"  {tok}")


if examples:
    show_example(examples[0])

[negative] (غذا و آشپزی, محاوره‌ای و غیررسمی) امروز برای شام یه خوراک لوبیای خوشمزه درست کردم.
  امروز
  برای
  شام
  یه
  خوراک
  لوبیای
  خوشمزه
  درست
  کردم
  .


## Save to JSONL

Same encoding as `generate_conditionals.ipynb`: one example per line with `id`, `tokens`, `tags`
(all `O`), `text`, `spans` (always empty), and `prompt_variant` (always `"negative"`), plus the
`system_prompt` text and the sampled `topic`/`register` used to build the prompt, written to a
separate output file so it can be merged with the positive dataset for training.

In [9]:
with open(CONFIG["output_path"], "w", encoding="utf-8") as f:
    for example in examples:
        f.write(json.dumps(example, ensure_ascii=False) + "\n")

print(f"Wrote {len(examples)} examples to {CONFIG['output_path']}")

Wrote 1000 examples to persian_negatives_bio.jsonl


## Quick sanity stats

In [10]:
num_with_tags = sum(any(t != "O" for t in ex["tags"]) for ex in examples)
avg_tokens = sum(len(ex["tokens"]) for ex in examples) / len(examples) if examples else 0

print(f"Examples with a non-O tag (should be 0): {num_with_tags}/{len(examples)}")
print(f"Average tokens per example:  {avg_tokens:.1f}")

Examples with a non-O tag (should be 0): 0/1000
Average tokens per example:  12.9
